# Lab 5F — SHAP Explainability

Per-prediction attribution for the deployed XGBoost model: **why** did the model predict this outcome for this client?

**Prerequisites:** Lab 3A has run, so a model version exists in the MLflow registry and the `training_data` Iceberg table is populated. Athena is the default data source; it falls back to a local CSV only if you flip `DATA_SOURCE = "csv"`.

**What you'll get:**
- Global feature importance — which features matter overall
- Per-prediction waterfall plots — why each decision
- Feature dependence plots — non-linear interactions
- SHAP artifacts logged back to MLflow (linked to the training run that produced the model)

> The interactive force plot in section 9.3 needs JavaScript. It renders in JupyterLab; static viewers such as the GitHub/GitLab renderer strip it.


## 1. Setup and Configuration

In [ ]:
# Load the shared .env written by lab0-setup/setup.ipynb, install this lab's
# package (pip -e) so `import src` resolves, and clear stale src modules.
from src.utils.notebook_env import bootstrap
env = bootstrap(install_package=True)

In [ ]:
import sys
import shap
print(f"Python: {sys.executable}")
print(f"SHAP version: {shap.__version__}")

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import matplotlib.pyplot as plt
import seaborn as sns

from src.config.config import MLFLOW_MONITORING_EXPERIMENT_NAME
from src.explainability import shap_analysis as sa

plt.style.use('default')
sns.set_palette("husl")
print("✓ All imports successful")

In [ ]:
sa.init_mlflow()

## 2. Load Model from MLflow

We'll load the latest registered model from MLflow. You can also specify a specific run ID.

In [ ]:
# Latest registered version of the model (search_model_versions, MLflow 3 API).
MODEL_RUN_ID = sa.find_latest_model_run()

In [ ]:
# Optional: Override with specific run ID
#MODEL_RUN_ID = "bbb5436e9bcb47e6ad7f4824f5df0c21"

model, feature_names = sa.load_model(MODEL_RUN_ID)

## 3. Load Training Data

**Data Source Choice**: This notebook defaults to loading from **Athena's `training_data` table** (not CSV).

**Why Athena is Recommended**:
- ✅ **Exact match**: `training_data` contains the POST-preprocessing data that the model was trained on
- ✅ **Already encoded**: Categorical columns like `customer_gender` are already label-encoded to integers
- ✅ **Accurate SHAP**: Using the exact training distribution ensures SHAP values accurately reflect model behavior
- ✅ **Always available**: No need to export/download CSV files manually

**CSV Option** (legacy):
- ⚠️ CSV path references an old file that's no longer auto-generated
- ⚠️ May require manual encoding to match preprocessing
- ⚠️ Only use if you have a specific reason to avoid Athena

**TL;DR**: Leave `DATA_SOURCE = "athena"` unless you have a compelling reason to change it.

In [ ]:
DATA_SOURCE = "athena"  # "athena" (RECOMMENDED — training_data Iceberg table, already preprocessed)
                        # "csv" (legacy — requires manual CSV export; not auto-generated)
NUM_SAMPLES = 5000

df = sa.load_training_sample(DATA_SOURCE, NUM_SAMPLES)

In [ ]:
# X follows the model's own feature order; features the model lists but the
# schema does not are zeroed (the printed metadata count should be 0).
X, y = sa.build_feature_matrix(df, feature_names)

## 4. Prepare Background Dataset

SHAP requires a background dataset to estimate the expected value (baseline prediction). We use stratified sampling to preserve the subscribed / not-subscribed ratio.

In [ ]:
BACKGROUND_SAMPLES = 500
EXPLAIN_SAMPLES = 200  # Number of predictions to explain

X_background, X_explain, y_explain = sa.split_background_and_explain(
    X, y, background_samples=BACKGROUND_SAMPLES, explain_samples=EXPLAIN_SAMPLES,
)

## 5. Compute SHAP Values

We use TreeExplainer, which is optimized for XGBoost and provides exact Shapley values.

In [ ]:
# check_additivity=True is useful for debugging, but slower.
explainer, shap_values, elapsed_time = sa.run_shap(model, X_background, X_explain)

## 6. Validate SHAP Values

Verify the mathematical consistency: `base_value + sum(shap_values) ≈ model_prediction`

In [ ]:
predictions = sa.predict_positive(model, X_explain)  # P(subscribed)
validation_results = sa.validate(explainer, shap_values, predictions, tolerance=1e-4)

## 7. Global Interpretability: Feature Importance

Which features are most important overall?

In [ ]:
feature_importance_df = sa.feature_importance(shap_values, feature_names, top_n=15)

## 8. SHAP Summary Plots

**Bar plot**: Shows feature importance (mean absolute SHAP)

**Beeswarm plot**: Shows feature value distribution and impact on predictions

In [ ]:
sa.plot_summary_bar(shap_values, X_explain)

In [ ]:
sa.plot_summary_beeswarm(shap_values, X_explain)

## 9. Local Interpretability: Individual Predictions

### 9.1 Waterfall Plot - Highest Positive-Prediction Case

In [ ]:
# The client the model is most confident will subscribe.
top_idx = sa.explain_prediction(explainer, shap_values, X_explain, predictions, y_explain, which="highest")

### 9.2 Waterfall Plot - Lowest Negative-Prediction Case

In [ ]:
# The client the model is most confident will NOT subscribe.
low_idx = sa.explain_prediction(explainer, shap_values, X_explain, predictions, y_explain, which="lowest")

### 9.3 Force Plot (Interactive)

In [ ]:
# Force plot for the likely-subscriber identified above (interactive HTML).
display(sa.force_plot(explainer, shap_values, X_explain, top_idx))

print("\n📊 Interactive force plot displayed above.")
print("  - Hover over features to see their names and values")
print("  - Red: Features increasing P(subscribed)")
print("  - Blue: Features decreasing P(subscribed)")

### 9.4 Decision Plot (Compare Multiple Predictions)

In [ ]:
sa.plot_decision(explainer, shap_values, X_explain, max_samples=50)

## 10. Feature Interactions: Dependence Plots

Dependence plots show how a feature's value affects its SHAP value, revealing interactions with other features.

In [ ]:
top_3_features = sa.plot_top_dependence(shap_values, X_explain, feature_names, feature_importance_df, top_k=3)

## 11. Save Artifacts

Save all plots, the SHAP values, and the feature-importance table under `shap_output/`.

In [ ]:
# Every plot plus shap_values.csv and feature_importance.csv, under shap_output/.
plot_paths, shap_csv_path = sa.save_artifacts(
    explainer, shap_values, X_explain, feature_names, feature_importance_df, output_dir="shap_output",
)

## 12. Log to MLflow

Log all SHAP artifacts to MLflow for tracking and comparison.

In [ ]:
shap_run_id = sa.log_to_mlflow(
    training_run_id=MODEL_RUN_ID,
    shap_values=shap_values,
    plot_paths=plot_paths,
    feature_names=feature_names,
    experiment_name=MLFLOW_MONITORING_EXPERIMENT_NAME,
)

## 13. Summary and Key Insights

In [ ]:
sa.print_summary(
    X_explain, X_background, feature_names, predictions, validation_results,
    elapsed_time, feature_importance_df, plot_paths, shap_csv_path, shap_run_id,
)